In [0]:
#Notebook 03: Raw to Bronze
#Reads latest successful Parquet file from Raw Volume
#Writes to Bronze Delta tables in overwrite mode (today's snapshot)


#Fetching values from job parameters
catalog_name  = dbutils.widgets.get("catalog_name")
meta_schema   = dbutils.widgets.get("meta_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")


In [0]:
#READING LATEST SUCCESSFUL FILE PER TABLE FROM CHILD METRICS

metrics_df = spark.sql(f"""
    SELECT table_name, file_location
    FROM (
        SELECT table_name, file_location,
               ROW_NUMBER() OVER (
                   PARTITION BY table_name
                   ORDER BY execution_time DESC
               ) as rn
        FROM {catalog_name}.{meta_schema}.child_metrics_metadata
        WHERE status = 'SUCCESS'
    )
    WHERE rn = 1
""")

tables = metrics_df.collect()
print(f"Tables to load into Bronze: {[row['table_name'] for row in tables]}\n")


#PROCESSING EACH TABLE: Raw Parquet to Bronze Delta

for row in tables:
    table_name    = row["table_name"]
    file_location = row["file_location"]

    try:
        #Step 1: Read Parquet from Raw Volume
        #raw_df = spark.read.parquet(file_location)
        raw_df = spark.read.format("delta").load(file_location) #for pipeline
        raw_row_count = raw_df.count()

        #Step 2: Write to Bronze as Delta in overwrite mode
        raw_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{catalog_name}.{bronze_schema}.{table_name.lower()}")

        #Step 3: Verify Bronze row count matches Raw
        bronze_row_count = spark.sql(f"""
            SELECT COUNT(*) as cnt
            FROM {catalog_name}.{bronze_schema}.{table_name.lower()}
        """).collect()[0]["cnt"]

        status = "Successful" if raw_row_count == bronze_row_count else "Unsuccessful"
        print(f"{status} {table_name} | Raw: {raw_row_count} | Bronze: {bronze_row_count}")

    except Exception as e:
        print(f"{table_name} FAILED: {str(e)}")

print("\nRaw to Bronze load completed...")

Tables to load into Bronze: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

Successful Album | Raw: 347 | Bronze: 347
Successful Artist | Raw: 275 | Bronze: 275
Successful Customer | Raw: 62 | Bronze: 62
Successful Employee | Raw: 8 | Bronze: 8
Successful Genre | Raw: 25 | Bronze: 25
Successful Invoice | Raw: 412 | Bronze: 412
Successful InvoiceLine | Raw: 2240 | Bronze: 2240
Successful MediaType | Raw: 5 | Bronze: 5
Successful Playlist | Raw: 18 | Bronze: 18
Successful PlaylistTrack | Raw: 8715 | Bronze: 8715
Successful Track | Raw: 3503 | Bronze: 3503

Raw to Bronze load completed...
